# National NBI Download

Downloads National Bridge Inventory (NBI) delimited ASCII files from FHWA for all 50 US states + DC, years 1992–2025, and saves one CSV per state to `raw/`.

**Source:** https://www.fhwa.dot.gov/bridge/nbi/ascii.cfm  
**URL pattern:** `https://www.fhwa.dot.gov/bridge/nbi/{year}/delimited/{STATE}{yy}.txt`

This is the national equivalent of `bridge_project.ipynb`'s download cell, which fetched MA only.
The per-state CSVs are the primary output. An optional final cell combines them into one file.

**Expected output sizes:**
- Per-state CSV: ~10–80 MB depending on state (TX, CA biggest)
- All states combined: ~15–20 GB — only combine if you have sufficient disk space and RAM

In [ ]:
import io
import os
import csv
import time
import requests
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
START_YEAR  = 1992
END_YEAR    = 2025
MAX_WORKERS = 8      # concurrent downloads; stay polite to FHWA server

RAW_DIR  = Path("raw")   # per-state CSVs land here
LOG_FILE = Path("download_log.txt")

BASE_URL = "https://www.fhwa.dot.gov/bridge/nbi/{year}/delimited/{state}{yy}.txt"
HEADERS  = {"User-Agent": "Mozilla/5.0 (research data download script)"}

RAW_DIR.mkdir(exist_ok=True)

In [ ]:
# ── State lookup: (postal abbreviation, FIPS code) ───────────────────────────
# FIPS codes match NBI STATE_CODE_001 field (zero-padded to 2 digits)
STATES = [
    ("AL", "01"), ("AK", "02"), ("AZ", "04"), ("AR", "05"), ("CA", "06"),
    ("CO", "08"), ("CT", "09"), ("DE", "10"), ("DC", "11"), ("FL", "12"),
    ("GA", "13"), ("HI", "15"), ("ID", "16"), ("IL", "17"), ("IN", "18"),
    ("IA", "19"), ("KS", "20"), ("KY", "21"), ("LA", "22"), ("ME", "23"),
    ("MD", "24"), ("MA", "25"), ("MI", "26"), ("MN", "27"), ("MS", "28"),
    ("MO", "29"), ("MT", "30"), ("NE", "31"), ("NV", "32"), ("NH", "33"),
    ("NJ", "34"), ("NM", "35"), ("NY", "36"), ("NC", "37"), ("ND", "38"),
    ("OH", "39"), ("OK", "40"), ("OR", "41"), ("PA", "42"), ("RI", "44"),
    ("SC", "45"), ("SD", "46"), ("TN", "47"), ("TX", "48"), ("UT", "49"),
    ("VT", "50"), ("VA", "51"), ("WA", "53"), ("WV", "54"), ("WI", "55"),
    ("WY", "56"),
]
print(f"States to download: {len(STATES)}")
print(f"Years: {START_YEAR}–{END_YEAR} ({END_YEAR - START_YEAR + 1} years)")
print(f"Total downloads: {len(STATES) * (END_YEAR - START_YEAR + 1):,}")

In [ ]:
# ── Download function (one state-year) ───────────────────────────────────────
def download_state_year(state_abbr: str, fips: str, year: int) -> tuple[object, int, str | None]:
    """
    Returns (df_or_None, row_count, error_message_or_None).
    Does not raise; all failures are returned as error strings.

    Parsing note: NBI delimited files use a single quote (') as the text
    qualifier, but text fields also contain literal apostrophes (e.g. "O'BRIEN",
    or a trailing "SR76''"). Passing quotechar="'" makes pandas treat those
    apostrophes as quote boundaries and desync the parser, raising
    "Expected 134 fields, saw 137". Verified across both the old (134-col) and
    new (123-col) schema years that NO field actually contains a comma, so we
    parse with QUOTE_NONE (split only on commas, treat quotes as literal text)
    and strip the decorative single quotes afterward.

    Rare exception: a few individual rows carry a genuine extra comma (e.g.
    PA 2024 line 9304 is one truly malformed record). QUOTE_NONE cannot merge a
    real extra delimiter, so rather than discard the whole file we retry with
    on_bad_lines="skip", dropping only the malformed row(s) and keeping every
    clean record. Verified to trigger on exactly one row in the whole 1,734-file
    corpus, so it never silently loses bulk data.
    """
    yy  = str(year)[-2:].zfill(2)
    url = BASE_URL.format(year=year, state=state_abbr, yy=yy)
    try:
        resp = requests.get(url, headers=HEADERS, timeout=90)
    except requests.RequestException as e:
        return None, 0, f"{year} network error: {e}"

    if resp.status_code == 404:
        return None, 0, f"{year} 404 (file not published)"
    if resp.status_code != 200:
        return None, 0, f"{year} HTTP {resp.status_code}"

    text = resp.content.decode("latin-1", errors="replace")
    try:
        df = pd.read_csv(io.StringIO(text), quoting=csv.QUOTE_NONE, low_memory=False)
    except Exception:
        # A genuine extra delimiter in one or more rows. Skip only those rows
        # (keeping every clean record) and report how many were dropped.
        n_data = text.rstrip("\n").count("\n")  # data rows (excludes header)
        try:
            df = pd.read_csv(io.StringIO(text), quoting=csv.QUOTE_NONE,
                             low_memory=False, on_bad_lines="skip")
        except Exception as e:
            return None, 0, f"{year} parse error: {e}"
        n_skipped = max(0, n_data - len(df))
        print(f"  {state_abbr} {year}: kept {len(df):,} rows, skipped {n_skipped} malformed row(s).")

    # Remove ONLY the surrounding single-quote qualifier — exactly what
    # quotechar="'" would have done. Critically, do NOT touch whitespace:
    # NBI pads keys like STRUCTURE_NUMBER_008 with leading spaces, and the
    # already-downloaded (quotechar-parsed) years keep that padding. Stripping
    # it here would make the same bridge's ID differ between old and re-fetched
    # years, splitting it into two "bridges" at the collapse-to-one-row step.
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip("'")

    df["STATE_FIPS"]  = fips        # explicit FIPS tag
    df["SURVEY_YEAR"] = year
    return df, len(df), None

In [ ]:
# ── Download all years for one state, save to raw/{STATE}_bridges.csv ────────
def download_state(state_abbr: str, fips: str) -> dict:
    out_path = RAW_DIR / f"{state_abbr}_bridges.csv"
    if out_path.exists():
        existing = pd.read_csv(out_path, low_memory=False)
        existing_years = set(existing["SURVEY_YEAR"].unique())
        target_years   = set(range(START_YEAR, END_YEAR + 1))
        missing_years  = sorted(target_years - existing_years)
        if not missing_years:
            print(f"  {state_abbr}: already complete, skipping.")
            return {"state": state_abbr, "rows": len(existing), "errors": []}
        print(f"  {state_abbr}: resuming — fetching {len(missing_years)} missing years.")
        years_to_fetch = missing_years
        existing_dfs   = [existing]
    else:
        years_to_fetch = list(range(START_YEAR, END_YEAR + 1))
        existing_dfs   = []

    all_dfs = list(existing_dfs)
    errors  = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {
            pool.submit(download_state_year, state_abbr, fips, yr): yr
            for yr in years_to_fetch
        }
        for future in as_completed(futures):
            result, n_rows, err = future.result()
            yr = futures[future]
            if err:
                errors.append(f"{state_abbr} {err}")
            else:
                all_dfs.append(result)

    if all_dfs:
        combined = pd.concat(all_dfs, ignore_index=True, sort=False)
        combined.to_csv(out_path, index=False)
        total = len(combined)
    else:
        total = 0

    return {"state": state_abbr, "rows": total, "errors": errors}

In [ ]:
# ── Main download loop ────────────────────────────────────────────────────────
# Downloads states sequentially (each state parallelises its years internally).
# Re-running this cell is safe — already-complete states are skipped.

all_errors  = []
state_stats = []

for i, (abbr, fips) in enumerate(STATES, 1):
    print(f"[{i}/{len(STATES)}] Downloading {abbr}...")
    t0     = time.time()
    result = download_state(abbr, fips)
    elapsed = time.time() - t0
    state_stats.append(result)
    all_errors.extend(result["errors"])
    status = f"  → {result['rows']:,} rows in {elapsed:.0f}s"
    if result["errors"]:
        status += f"  ({len(result['errors'])} year(s) failed)"
    print(status)

# Write log
with open(LOG_FILE, "w") as f:
    if all_errors:
        f.write("\n".join(all_errors))
    else:
        f.write("No errors.")

print(f"\nDone. {len(all_errors)} total errors logged to {LOG_FILE}.")

In [ ]:
# ── Sanity checks ────────────────────────────────────────────────────────────
stats_df = pd.DataFrame(state_stats)
print("Per-state row counts:")
print(stats_df[["state", "rows"]].sort_values("rows", ascending=False).to_string(index=False))
print(f"\nTotal rows across all states: {stats_df['rows'].sum():,}")
print(f"Expected: ~21,000,000 (620K bridges × 34 years)")

In [ ]:
# ── Cross-check MA against the existing MA-only file ─────────────────────────
# Compare our MA download to the already-complete massachusetts_bridges_all_years.csv

ma_new = pd.read_csv(RAW_DIR / "MA_bridges.csv", low_memory=False)
ma_ref = pd.read_csv("../massachusetts_bridges_all_years.csv", low_memory=False)

print(f"New MA download : {len(ma_new):,} rows, {ma_new['STRUCTURE_NUMBER_008'].nunique():,} unique bridges")
print(f"Reference MA    : {len(ma_ref):,} rows, {ma_ref['STRUCTURE_NUMBER_008'].nunique():,} unique bridges")
print()

new_per_year = ma_new.groupby("SURVEY_YEAR").size().rename("new")
ref_per_year = ma_ref.groupby("SURVEY_YEAR").size().rename("ref")
comparison   = pd.concat([new_per_year, ref_per_year], axis=1)
comparison["diff"] = comparison["new"] - comparison["ref"]
print("Year-by-year comparison (diff = new − reference):")
print(comparison.to_string())

In [ ]:
# ── (Optional) Combine all states into one file ───────────────────────────────
# WARNING: expect ~15–20 GB on disk and requires ~30–60 GB RAM to hold in memory.
# Skip this cell if you plan to process states individually in later pipeline steps.

# Uncomment to run:
# combined_chunks = []
# for abbr, _ in STATES:
#     path = RAW_DIR / f"{abbr}_bridges.csv"
#     if path.exists():
#         combined_chunks.append(pd.read_csv(path, low_memory=False))
#         print(f"  Loaded {abbr}")
#
# combined = pd.concat(combined_chunks, ignore_index=True, sort=False)
# combined.to_csv("us_bridges_all_years.csv", index=False)
# print(f"Saved us_bridges_all_years.csv ({len(combined):,} rows)")